In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exp_raw_data2 = '/content/drive/MyDrive/Colab Notebooks/Dataset_modelB.xlsx'
collab_data = pd.read_excel(exp_raw_data2)

In [ ]:
collab_data.corr().abs()['totalCr_concentration']

In [ ]:
collab_data.columns

In [ ]:
import seaborn as sns
# Define input and output features
input_features = ['colour_a', 'colour_b', 'colour_L', 'conductivity']
output_features = ['totalCr_concentration', 'CrVI_concentration']

# Create the subplot grid
fig, axes = plt.subplots(len(output_features), len(input_features), figsize=(4*len(input_features), 4*len(output_features)))

# Loop over rows and columns to create scatter plots
for i, y_feat in enumerate(output_features):
    for j, x_feat in enumerate(input_features):
        ax = axes[i, j] if len(output_features) > 1 else axes[j]  # Handle 1-row case
        sns.scatterplot(data=collab_data, x=x_feat, y=y_feat, ax=ax)
        ax.set_xlabel(x_feat)
        ax.set_ylabel(y_feat)

plt.tight_layout()
plt.savefig('custom_pairplot_modelB.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
def remove_outliers_iqr(df, column, k=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - k * IQR
    upper_bound = Q3 + k * IQR
    outliers_removed = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers_removed, outliers

columns_for_outlier_removal = ['totalCr_concentration','colour_a','colour_b','colour_L']

outliers_removed_data = collab_data.copy()
outliers_data = pd.DataFrame(columns=collab_data.columns)

for column in columns_for_outlier_removal:
    outliers_removed_data, outliers = remove_outliers_iqr(outliers_removed_data, column)
    outliers_data = pd.concat([outliers_data, outliers])

print('Outliers list length', len(outliers_data))
print('Corrected dataset length', len(outliers_removed_data))

In [ ]:
outliers_removed_data.corr().abs()['totalCr_concentration']

In [ ]:
y = outliers_removed_data['totalCr_concentration']
features = ['colour_a','colour_b','colour_L','conductivity']
X = outliers_removed_data[features]

X_testing, X_train, y_testing, y_train = train_test_split(X, y, test_size=0.7, random_state=27)
X_val, X_test, y_val, y_test = train_test_split(X_testing, y_testing, test_size=0.5, random_state=27)
print("Training set size", len(X_train))
print("Validation set size", len(X_val))
print("Training set size", len(X_test))
drive_path1 = '/content/drive/My Drive/Colab Notebooks/X_testB_2.csv'
X_test.to_csv(drive_path1, index=False)
drive_path2 = '/content/drive/My Drive/Colab Notebooks/y_testB_2.csv'
y_test.to_csv(drive_path2, index=False)

In [ ]:
outliers_removed_data['colour_a'].plot(kind='kde')
outliers_removed_data['colour_a'].plot(kind='hist', density = True)

In [ ]:
outliers_removed_data['colour_b'].plot(kind='kde')
outliers_removed_data['colour_b'].plot(kind='hist', density = True)

In [ ]:
hist = outliers_removed_data['totalCr_concentration'].plot(kind='hist', density = True, bins=5)

In [ ]:
train_data = pd.concat([X_train, y_train], axis=1)

majority_class_train = train_data[train_data['totalCr_concentration'] <= 12.5]
minority_class_train = train_data[train_data['totalCr_concentration'] > 12.5]

downsampled_majority_train = resample(majority_class_train,
                                      replace=False,
                                      n_samples=len(minority_class_train),
                                      random_state=27)

balanced_train_data = pd.concat([downsampled_majority_train, minority_class_train])

X_train_balanced = balanced_train_data[features]
y_train_balanced = balanced_train_data['totalCr_concentration']

print("Majority class size:", len(majority_class_train))
print("Minority class size:", len(minority_class_train))
print("Original training dataset size:", len(X_train))
print("Balanced training dataset size:", len(X_train_balanced))

In [ ]:
from sklearn.linear_model import LinearRegression

# Example DataFrame

# Fit the linear regression model
model = LinearRegression()
model.fit(X_train_balanced, y_train_balanced)
model_prediction = model.predict(X_train_balanced)

# Output results
print("Intercept:", model.intercept_)
print("Coefficients:", dict(zip(X.columns, model.coef_)))
print("R² Score:", model.score(X, y))  # Coefficient of determination

In [ ]:
model_mae_TRAIN = mean_absolute_error(model_prediction, y_train_balanced)
model_mape_TRAIN = mean_absolute_percentage_error(model_prediction, y_train_balanced)
model_linearity_TRAIN = r2_score(model_prediction, y_train_balanced)

print('rf_trail1_mae_TRAIN =', model_mae_TRAIN)
print('rf_trail1_mape_TRAIN =', model_mape_TRAIN)
print('rf_trail1_linearity_TRAIN =', model_linearity_TRAIN)

plt.figure(figsize=(10, 6))
plt.scatter(y_train_balanced, model_prediction, color='blue', label='Predicted vs Actual')
plt.plot([min(y_train_balanced), max(y_train_balanced)], [min(y_train_balanced), max(y_train_balanced)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.xlabel('Actual Values', fontsize=16)
plt.ylabel('Predicted Values',fontsize=16)
# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.legend(fontsize=14)
plt.savefig('linear_model_B_totalCr_TRAIN.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
val_prediction = model.predict(X_val)

# Output results

model_mae_VAL = mean_absolute_error(val_prediction, y_val)
model_mape_VAL = mean_absolute_percentage_error(val_prediction, y_val)
model_linearity_VAL = r2_score(val_prediction, y_val)

print('rf_trail1_mae_TRAIN =', model_mae_VAL)
print('rf_trail1_mape_TRAIN =', model_mape_VAL)
print('rf_trail1_linearity_TRAIN =', model_linearity_VAL)

plt.figure(figsize=(10, 6))
plt.scatter(y_val, val_prediction, color='blue', label='Predicted vs Actual')
plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.xlabel('Actual Values', fontsize=16)
plt.ylabel('Predicted Values',fontsize=16)

# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.legend(fontsize=14)
plt.savefig('linear_model_B_totalCr_VAL.svg', format='svg', bbox_inches='tight')
plt.show()


In [ ]:
rf_trail1 = RandomForestRegressor(random_state=27)
rf_trail1.fit(X_train_balanced, y_train_balanced)

rf_trail1_prediction_train = rf_trail1.predict (X_train_balanced)

rf_trail1_mae_TRAIN = mean_absolute_error(rf_trail1_prediction_train, y_train_balanced)
rf_trail1_mape_TRAIN = mean_absolute_percentage_error(rf_trail1_prediction_train, y_train_balanced)
rf_trail1_linearity_TRAIN = r2_score(rf_trail1_prediction_train, y_train_balanced)

print('rf_trail1_mae_TRAIN =', rf_trail1_mae_TRAIN)
print('rf_trail1_mape_TRAIN =', rf_trail1_mape_TRAIN)
print('rf_trail1_linearity_TRAIN =', rf_trail1_linearity_TRAIN)

plt.figure(figsize=(10, 6))
plt.scatter(y_train_balanced, rf_trail1_prediction_train, color='blue', label='Predicted vs Actual')
plt.plot([min(y_train_balanced), max(y_train_balanced)], [min(y_train_balanced), max(y_train_balanced)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.title('Prediction vs Actual')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.legend()
plt.show()

In [ ]:
rf_trail1_prediction = rf_trail1.predict (X_val)
rf_trail1_mae = mean_absolute_error(rf_trail1_prediction, y_val)
rf_trail1_mape = mean_absolute_percentage_error(rf_trail1_prediction, y_val)
rf_trail1_linearity = r2_score(rf_trail1_prediction, y_val)

print('rf_trail1_mae =', rf_trail1_mae)
print('rf_trail1_mape =', rf_trail1_mape)
print('rf_trail1_linearity =', rf_trail1_linearity)

plt.figure(figsize=(10, 6))
plt.scatter(y_val, rf_trail1_prediction, color='blue', label='Predicted vs Actual')
plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.title('Prediction vs Actual')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.legend()
plt.show()

In [ ]:
feat_importance_model_rf_trail1 = pd.Series(rf_trail1.feature_importances_, index=X_train_balanced.columns)
feat_importance_model_rf_trail1.nlargest(5).plot(kind='barh')

Hyperparameter optimization (bayesian optimization)

In [ ]:
!pip install bayesian-optimization scikit-learn pandas
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from bayes_opt import BayesianOptimization
from scipy.stats import pearsonr

In [ ]:
def linearity_score(y_true, y_pred):
       residuals = y_true - y_pred
       corr_coef, _ = pearsonr(y_pred, residuals)
       return np.abs(corr_coef)

def objective(n_estimators, max_depth, min_samples_leaf, max_features):
    rf_trail3= RandomForestRegressor(n_estimators=int(n_estimators),
                                  max_depth=int(max_depth),
                                  min_samples_leaf=int(min_samples_leaf),
                                  max_features=min(max_features, 0.999),
                                  random_state=27)
    y_pred = cross_val_predict(rf_trail3, X_train_balanced, y_train_balanced, cv=5)
    mse = mean_squared_error(y_train_balanced, y_pred)
    linearity = linearity_score(y_train_balanced, y_pred)
    score = -mse - linearity
    return score

In [ ]:
param_bounds = {
    'n_estimators': (10, 100),
    'max_depth': (1, 50),
    'min_samples_leaf': (1,5),
    'max_features': (0.1, 0.999),
}
optimizer = BayesianOptimization(f=objective, pbounds=param_bounds, random_state=27)
optimizer.maximize(init_points=5, n_iter=15)

In [ ]:
best_params=optimizer.max['params']
best_params

In [ ]:
rf_trail3_bayesian_model = RandomForestRegressor(n_estimators=99,
                                   max_depth=9,
                                   min_samples_leaf=1,
                                   max_features=0.78,
                                   random_state=27)
rf_trail3_prediction_train = rf_trail3_bayesian_model.fit(X_train_balanced, y_train_balanced)
rf_trail3_prediction_train = rf_trail3_bayesian_model.predict(X_train_balanced)

rf_trail3_mae_TRAIN = mean_absolute_error(rf_trail3_prediction_train, y_train_balanced)
rf_trail3_mape_TRAIN = mean_absolute_percentage_error(rf_trail3_prediction_train, y_train_balanced)
rf_trail3_linearity_TRAIN = r2_score(rf_trail3_prediction_train, y_train_balanced)

print('rf_trail3_mae_TRAIN =', rf_trail3_mae_TRAIN)
print('rf_trail3_mape_TRAIN =', rf_trail3_mape_TRAIN)
print('rf_trail3_linearity_TRAIN =', rf_trail3_linearity_TRAIN)

plt.figure(figsize=(10, 6))
plt.scatter(y_train_balanced, rf_trail3_prediction_train, color='blue', label='Predicted vs Actual')
plt.plot([min(y_train_balanced), max(y_train_balanced)], [min(y_train_balanced), max(y_train_balanced)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.xlabel('Actual Values', fontsize=16)
plt.ylabel('Predicted Values',fontsize=16)

# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.legend(fontsize=14)

plt.savefig('B_totalCr_train.svg', format='svg')

plt.show()



In [ ]:
rf_trail3_prediction = rf_trail3_bayesian_model.predict (X_val)

rf_trail3_mae = mean_absolute_error(rf_trail3_prediction, y_val)
rf_trail3_mape = mean_absolute_percentage_error(rf_trail3_prediction, y_val)
rf_trail3_linearity = r2_score(rf_trail3_prediction, y_val)

print('rf_trail2_mae/rf_trail3_mae =',  rf_trail3_mae)
print('rf_trail2_mape/rf_trail3_mape =',  rf_trail3_mape)
print('rf_trail2_linearity/rf_trail3_linearity =', rf_trail3_linearity)

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(y_val, rf_trail3_prediction, color='blue', label='Predicted vs Actual')
plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], color='red', linestyle='--', label='Perfect Prediction Line')
plt.xlabel('Actual Values', fontsize=16)
plt.ylabel('Predicted Values',fontsize=16)

# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.legend(fontsize=14)
plt.savefig('B_totalCr_validation.svg', format='svg')
plt.show()



In [ ]:
X_train_balanced.columns

In [ ]:
feat_importance_model_rf_trail3 = pd.Series(rf_trail3_bayesian_model.feature_importances_, index=['colour_a', 'colour_b', 'colour_L', 'conductivity'])

# Start a new figure
plt.figure()

# Plot the top 5 features as a horizontal bar chart
feat_importance_model_rf_trail3.nlargest(5).plot(kind='barh')

# Save the plot to an SVG file
plt.savefig('B_totalCr_featureimp_v2.svg', format='svg', bbox_inches='tight')

# Optional: show the plot
plt.show()

Final model saving

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from joblib import dump
v2_rf_optimized_modelB_totalCr = rf_trail3_bayesian_model
drive_path = '/content/drive/My Drive/Colab Notebooks/v2_rf_optimized_modelB_totalCr.joblib'
dump(v2_rf_optimized_modelB_totalCr, drive_path)
print("Model saved successfully to Google Drive.")